# Trimorious M1.2 — Scientific Dataset Remediation
**Candidate-only • Audit-first • No training • No publishing**

This notebook reads the existing D01–D04 datasets and creates a separate candidate workspace. It does not overwrite, push, or modify any original Hugging Face or GitHub repository.

### M1.2 goals
1. Reconcile source counts and account for every record.
2. Normalize schemas while retaining original raw records.
3. Detect exact duplicates and flag near duplicates for review.
4. Triage D01 scientific-context relevance and D02 mathematical tasks.
5. Apply limited, transparent SymPy checks where parsing is safe.
6. Preserve D03 as a retrieval corpus, not automatic SFT data.
7. Quarantine D04 until tool traces can be replayed and validated.
8. Require human approval before candidate train/validation exports.
9. Keep D07 evaluation material separate and never train on it.

Automated flags are triage signals, not scientific validation.

## 0. Runtime controls

Run in Google Colab. Source datasets are read-only inputs. All outputs go under `/content/trimorious_m1_2/`. Do not put access tokens in notebook cells.

In [ ]:
%pip -q install datasets huggingface_hub pandas pyarrow rapidfuzz sympy scikit-learn tqdm

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import pandas as pd, numpy as np
import json, re, hashlib, zipfile, shutil
from rapidfuzz import fuzz
from datasets import load_dataset
import sympy as sp

ROOT = Path("/content/trimorious_m1_2")
INPUTS = ROOT / "inputs"
OUT = ROOT / "output"
REPORTS = OUT / "reports"
CANDIDATES = OUT / "candidate_datasets"
for p in [INPUTS, REPORTS, CANDIDATES]:
    p.mkdir(parents=True, exist_ok=True)

ALLOW_TRAINING = False
ALLOW_PUBLISH = False
AUTO_ACCEPT = False

RUN_ID = datetime.now(timezone.utc).strftime("m12_%Y%m%dT%H%M%SZ")
SOURCE_REPOS = {
    "D01": "roime81/trimorious-scientific-instructions",
    "D02": "roime81/trimorious-symbolic-trajectories",
    "D03": "roime81/trimorious-knowledge-base",
    "D04": "roime81/trimorious-agentic-reasoning-knowledge",
}
print("Run:", RUN_ID)

## 1. Load source datasets

The default reads the public Hugging Face repositories. If a dataset has multiple splits, all splits are combined while preserving `_source_split`. To use local exports instead, set `USE_LOCAL_FILES=True` and fill `LOCAL_FILES`.

In [ ]:
USE_LOCAL_FILES = False
LOCAL_FILES = {}  # e.g. {"D01": "/content/trimorious_m1_2/inputs/D01.jsonl"}

def read_local(path):
    path = Path(path)
    if path.suffix.lower() in [".jsonl", ".ndjson"]:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        obj = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(obj, dict):
            for key in ["data", "records", "examples", "train"]:
                if isinstance(obj.get(key), list):
                    obj = obj[key]
                    break
        return pd.DataFrame(obj if isinstance(obj, list) else [obj])
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    raise ValueError("Unsupported local file: " + str(path))

raw, load_errors = {}, {}
for name, repo in SOURCE_REPOS.items():
    try:
        if USE_LOCAL_FILES:
            raw[name] = read_local(LOCAL_FILES[name])
        else:
            loaded = load_dataset(repo)
            pieces = []
            for split_name, split in loaded.items():
                frame = split.to_pandas()
                frame["_source_split"] = split_name
                pieces.append(frame)
            raw[name] = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
        print(name, len(raw[name]), "rows")
    except Exception as e:
        load_errors[name] = repr(e)
        print("LOAD ERROR", name, repr(e))

## 2. Inventory and historical count reconciliation

The previous audit recorded D01=3,706, D02=7,784, D03=7,784 and D04=225. D01+D02+D04 equals 11,715, while the prior normalization report showed 11,119. This notebook recalculates counts from the currently loaded source data; it must not assume that the historical numbers are still current.

In [ ]:
inventory = []
for name, frame in raw.items():
    inventory.append({
        "dataset": name, "rows_loaded": len(frame),
        "columns": len(frame.columns),
        "column_names": ", ".join(map(str, frame.columns)),
        "empty_rows": int(frame.isna().all(axis=1).sum()) if len(frame) else 0,
        "load_error": load_errors.get(name, "")
    })
inventory = pd.DataFrame(inventory)
display(inventory)
inventory.to_csv(REPORTS / "source_inventory.csv", index=False)
print("Current D01+D02+D04:", sum(len(raw.get(k, [])) for k in ["D01","D02","D04"]))
print("Current D03 retrieval rows:", len(raw.get("D03", [])))

## 3. Schema mapping and normalization

Inspect the columns printed below. If automatic field mapping is incorrect, edit `FIELD_OVERRIDES` and rerun this section. Do not proceed until `instruction` and `output` map to the intended fields. Every normalized row retains its complete original record as JSON.

In [ ]:
ALIASES = {
 "instruction": ["instruction","prompt","question","query","problem","task"],
 "input": ["input"],
 "output": ["output","response","answer","solution","completion","target"],
 "context": ["context","scientific_context","paper_context","background","source_text"],
 "source": ["source","source_title","paper_title","title","document_title"],
 "source_url": ["source_url","url","doi_url","doi"],
 "license": ["license","license_id","data_license"],
 "task_type": ["task_type","type","category"],
 "domain": ["domain","field","discipline","subject"],
 "tool_trace": ["tool_trace","trace","tool_calls","trajectory","actions"],
 "verification": ["verification","verified","verification_result","sympy_check"]
}
FIELD_OVERRIDES = {
 # Example: "D01": {"instruction":"question", "output":"answer", "context":"paper_context"}
}
def find_col(cols, aliases):
    lookup = {str(c).lower(): c for c in cols}
    return next((lookup[a] for a in aliases if a in lookup), None)

def txt(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return ""
    if isinstance(x, (dict,list)): return json.dumps(x, ensure_ascii=False, default=str)
    return re.sub(r"\s+", " ", str(x).replace("\x00"," ")).strip()

normalized, mappings = {}, []
for name, frame in raw.items():
    out = pd.DataFrame(index=frame.index)
    mapping = {"dataset": name}
    for field, aliases in ALIASES.items():
        col = FIELD_OVERRIDES.get(name, {}).get(field) or find_col(frame.columns, aliases)
        mapping[field] = col
        out[field] = frame[col].map(txt) if col else ""
    out["source_dataset"] = name
    out["source_row_id"] = [f"{name}:{i}" for i in range(len(frame))]
    out["raw_record_json"] = frame.apply(lambda r: json.dumps(r.to_dict(), ensure_ascii=False, default=str), axis=1)
    normalized[name] = out.reset_index(drop=True)
    mappings.append(mapping)

display(pd.DataFrame(mappings))
pd.DataFrame(mappings).to_json(REPORTS / "schema_mapping.json", orient="records", indent=2)
for name, frame in raw.items():
    print("\n", name, "columns:", list(frame.columns))
    display(frame.head(2))

## 4. Exact duplicate detection and disposition ledger

D01, D02 and D04 are included in the SFT-source reconciliation pool. D03 is intentionally excluded from that pool. Exact duplicates are flagged, not silently deleted; the original records remain preserved.

In [ ]:
def row_hash(row):
    payload = "\n".join([row.get("instruction",""), row.get("input",""), row.get("output","")]).lower()
    payload = re.sub(r"\W+", " ", payload).strip()
    return hashlib.sha256(payload.encode()).hexdigest()

pool = pd.concat([normalized[k] for k in ["D01","D02","D04"] if k in normalized], ignore_index=True)
pool["exact_hash"] = pool.apply(row_hash, axis=1)
pool["is_exact_duplicate"] = pool.duplicated("exact_hash", keep="first")
pool["initial_disposition"] = np.where(pool["is_exact_duplicate"], "duplicate_candidate", "needs_human_review")
pool.to_parquet(OUT / "normalized_sft_sources.parquet", index=False)
print("SFT source rows:", len(pool))
print("Unique exact fingerprints:", pool["exact_hash"].nunique())
print("Exact duplicate rows:", int(pool["is_exact_duplicate"].sum()))

## 5. Near-duplicate triage

Bounded pairwise comparison to avoid excessive Colab runtime. The output is a review list, not an automatic deletion list. Increase the cap only if runtime permits.

In [ ]:
NEAR_DUPLICATE_CAP = 2500
near = []
for name in ["D01","D02","D04"]:
    sub = pool[(pool.source_dataset == name) & (~pool.is_exact_duplicate)].head(NEAR_DUPLICATE_CAP)
    recs = sub[["source_row_id","instruction"]].to_dict("records")
    for i, a in enumerate(recs):
        if not a["instruction"]: continue
        for j in range(i+1, min(i+40, len(recs))):
            b = recs[j]
            score = fuzz.token_set_ratio(a["instruction"], b["instruction"])
            if score >= 96:
                near.append({"dataset":name,"row_a":a["source_row_id"],"row_b":b["source_row_id"],"similarity":score})
near_df = pd.DataFrame(near)
near_df.to_csv(REPORTS / "near_duplicate_candidates.csv", index=False)
print("Near duplicate pairs flagged:", len(near_df))

## 6. D01 context/task relevance triage

The earlier audit found examples where generic mathematics was paired with unrelated scientific context, followed by unsupported claims connecting the math to the paper. This section flags possible mismatches using simple lexical features. It cannot establish semantic relevance; expert review remains mandatory.

In [ ]:
SCIENCE_TERMS = set("experiment hypothesis measurement dataset model method sample clinical patient biological chemical physics astronomy climate neural network protein gene cell material energy spectrum regression correlation causal statistical simulation".split())
def words(s):
    return set(re.findall(r"[a-zA-Z]{3,}", txt(s).lower()))

def d01_features(row):
    ctx, task = row.get("context",""), row.get("instruction","") + " " + row.get("output","")
    ct, tt = words(ctx), words(task)
    overlap = len(ct & tt) / max(1, len(tt))
    generic_math = bool(re.search(r"\b(derivative|differentiate|integral|solve for x|factorise|factorize|simplify)\b", row.get("instruction","").lower()))
    flag = "review_context_task_mismatch" if len(ctx) > 80 and generic_math and overlap < .12 else ("review_low_overlap" if len(ctx)>80 and overlap < .05 else "no_obvious_lexical_mismatch")
    return pd.Series({"context_present":len(ctx)>80,"science_context_terms":len(ct & SCIENCE_TERMS),"task_context_overlap":round(overlap,4),"generic_math_task":generic_math,"relevance_flag":flag})

d01 = normalized.get("D01", pd.DataFrame()).copy()
if len(d01):
    d01 = pd.concat([d01, d01.apply(d01_features, axis=1)], axis=1)
    d01.to_parquet(CANDIDATES / "D01_relevance_triage.parquet", index=False)
    display(d01.relevance_flag.value_counts())
else:
    print("D01 unavailable")

## 7. D02 task classification and limited SymPy reference calculations

Classes: `pure_math`, `scientific_model`, `research_derivation`, `numerical_or_symbolic_check`, `formal_proof`. The SymPy parser below only supports a narrow pattern of derivative/integral prompts. It does not execute arbitrary code from dataset records.

In [ ]:
def classify_math(row):
    s = (row.get("instruction","") + " " + row.get("input","")).lower()
    if re.search(r"\b(prove|proof|show that|theorem)\b", s): return "formal_proof"
    if row.get("context") and len(row["context"]) > 80 and re.search(r"\b(model|equation|derive|parameter|phenomenon)\b", s): return "scientific_model"
    if re.search(r"\b(derive|derivation|from the paper|according to)\b", s): return "research_derivation"
    if re.search(r"\b(derivative|differentiate|integral|integrate|solve|simplify)\b", s): return "numerical_or_symbolic_check"
    return "pure_math"

def sympy_reference(instruction):
    patterns = [
      (r"(?:differentiate|derivative of)\s+(.+?)\s+(?:with respect to|w\.r\.t\.)\s+([a-zA-Z])", "derivative"),
      (r"(?:integrate|integral of)\s+(.+?)\s+(?:with respect to|w\.r\.t\.)\s+([a-zA-Z])", "integral")
    ]
    for pattern, kind in patterns:
        m = re.search(pattern, instruction, re.I)
        if m:
            expr_text, var = m.group(1), m.group(2)
            try:
                symbol = sp.Symbol(var)
                expr = sp.sympify(expr_text, locals={var:symbol})
                result = sp.diff(expr,symbol) if kind == "derivative" else sp.integrate(expr,symbol)
                return {"status":"computed_reference_only","kind":kind,"reference":str(result)}
            except Exception as e:
                return {"status":"not_checked","reason":"parse_failed: "+type(e).__name__}
    return {"status":"not_checked","reason":"outside_supported_pattern"}

d02 = normalized.get("D02", pd.DataFrame()).copy()
if len(d02):
    d02["math_task_class"] = d02.apply(classify_math, axis=1)
    checks = d02.instruction.map(sympy_reference)
    d02["sympy_status"] = [x["status"] for x in checks]
    d02["sympy_reference"] = [x.get("reference","") for x in checks]
    d02["sympy_check_kind"] = [x.get("kind","") for x in checks]
    d02["sympy_reason"] = [x.get("reason","") for x in checks]
    d02.to_parquet(CANDIDATES / "D02_math_triage.parquet", index=False)
    display(d02.math_task_class.value_counts())
    display(d02.sympy_status.value_counts())
else:
    print("D02 unavailable")

### SymPy interpretation warning

`computed_reference_only` means a reference expression was computed from a recognized prompt fragment. It does **not** mean the supplied answer matches it. It does not validate derivation steps, assumptions, units, boundary conditions, scientific relevance or proof correctness. A human reviewer must compare answer and reference.

## 8. D04 agentic/tool-use quarantine

D04 remains quarantined until actual tool traces can be replayed. A narrative saying a tool was used is not sufficient. Review whether each record includes tool identity, arguments, actual tool output, sequence and final answer grounded in that output.

In [ ]:
d04 = normalized.get("D04", pd.DataFrame()).copy()
if len(d04):
    def audit_trace(row):
        try: obj = json.loads(row["raw_record_json"])
        except Exception: obj = {}
        keys = {str(k).lower() for k in obj} if isinstance(obj,dict) else set()
        trace = row.get("tool_trace","").lower()
        has_tool = bool(row.get("tool_trace")) or any("tool" in k for k in keys)
        has_args = any("arg" in k for k in keys) or "arguments" in trace
        has_result = any(("result" in k or "output" in k) for k in keys) or "tool output" in trace
        has_sequence = bool(row.get("tool_trace")) or any(("step" in k or "sequence" in k) for k in keys)
        return pd.Series({"has_tool":has_tool,"has_arguments":has_args,"has_result":has_result,"has_sequence":has_sequence,
          "d04_disposition":"manual_trace_replay_required" if all([has_tool,has_args,has_result,has_sequence]) else "quarantine_incomplete_trace"})
    d04 = pd.concat([d04,d04.apply(audit_trace,axis=1)],axis=1)
    d04.to_parquet(CANDIDATES / "D04_quarantine.parquet",index=False)
    display(d04.d04_disposition.value_counts())
else:
    print("D04 unavailable")

## 9. D03 retrieval-only treatment and provenance

D03 is not converted into SFT examples. It is retained as a retrieval-corpus candidate pending source provenance, license, extraction quality and chunk-level review. Missing license metadata is unresolved; it is not permission to redistribute.

In [ ]:
d03 = normalized.get("D03", pd.DataFrame()).copy()
if len(d03):
    d03["provenance_status"] = d03.apply(
      lambda r: "metadata_present_review_license" if r["source"] and r["source_url"] and r["license"] else "incomplete_provenance_review_required", axis=1)
    d03.to_parquet(CANDIDATES / "D03_retrieval_candidate_unapproved.parquet",index=False)
    display(d03.provenance_status.value_counts())
else:
    print("D03 unavailable")

## 10. Human review queue

No automatic acceptance. Complete `human_decision` manually using: `accept_candidate`, `revise`, `reject`, `needs_domain_expert`, or `needs_source_permission`. For D01, prioritize context/task mismatches. For D02, inspect derivations, scientific models and proof tasks. For D04, require trace replay.

In [ ]:
review_parts = []
for name, frame in [("D01",d01),("D02",d02),("D04",d04)]:
    if len(frame):
        tmp = frame.copy()
        if name == "D01": tmp["review_reason"] = tmp["relevance_flag"]
        elif name == "D02": tmp["review_reason"] = tmp["math_task_class"]
        else: tmp["review_reason"] = tmp["d04_disposition"]
        tmp["human_decision"] = ""
        tmp["reviewer_notes"] = ""
        review_parts.append(tmp)
review_queue = pd.concat(review_parts,ignore_index=True) if review_parts else pd.DataFrame()
review_path = REPORTS / "human_review_queue.csv"
review_queue.to_csv(review_path,index=False)
print("Review queue:", len(review_queue), "rows")
print("Saved:", review_path)
display(review_queue.head(8))

## 11. Source-group split function

When approved records are later exported, split by source URL/title rather than random rows to reduce leakage. Records without source metadata fall back to their normalized task fingerprint and should be reviewed carefully. This function does not run until the human review file is completed.

In [ ]:
def split_by_source(approved, validation_fraction=0.10, seed=42):
    def group_id(r):
        key = r.get("source_url","") or r.get("source","")
        if not key: key = row_hash(r)
        return hashlib.sha256(str(key).lower().encode()).hexdigest()
    frame = approved.copy()
    frame["split_group"] = frame.apply(group_id,axis=1)
    groups = frame.split_group.drop_duplicates().tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    n_val = max(1, round(len(groups)*validation_fraction)) if len(groups)>1 else 0
    val_groups = set(groups[:n_val])
    frame["candidate_split"] = frame.split_group.map(lambda x: "validation" if x in val_groups else "train")
    train = frame[frame.candidate_split=="train"].copy()
    val = frame[frame.candidate_split=="validation"].copy()
    assert set(train.split_group).isdisjoint(set(val.split_group))
    return train,val

## 12. Export reviewed candidate datasets (manual gate)

Download `human_review_queue.csv`, complete the decision and reviewer-notes columns, then upload the reviewed file back to Colab. Only records explicitly marked `accept_candidate` from D01/D02 are eligible for the initial SFT candidate. D04 remains excluded until separately validated. The output is versioned and separate from original repositories.

In [ ]:
REVIEWED_QUEUE_PATH = ""  # e.g. "/content/trimorious_m1_2/reports/human_review_queue_reviewed.csv"

def build_candidate(reviewed_path):
    if not reviewed_path or not Path(reviewed_path).exists():
        raise RuntimeError("Upload the manually reviewed queue and set REVIEWED_QUEUE_PATH.")
    reviewed = pd.read_csv(reviewed_path).fillna("")
    approved = reviewed[
      (reviewed.human_decision.astype(str).str.strip()=="accept_candidate") &
      (reviewed.source_dataset.isin(["D01","D02"]))
    ].copy()
    approved = approved[(approved.instruction.str.len()>0)&(approved.output.str.len()>0)]
    if approved.empty: raise RuntimeError("No eligible D01/D02 records approved.")
    train,val = split_by_source(approved)
    train_path = CANDIDATES / f"D01_D02_candidate_train_{RUN_ID}.jsonl"
    val_path = CANDIDATES / f"D01_D02_candidate_validation_{RUN_ID}.jsonl"
    train.to_json(train_path,orient="records",lines=True,force_ascii=False)
    val.to_json(val_path,orient="records",lines=True,force_ascii=False)
    return train_path,val_path,len(train),len(val)

# After completing manual review, uncomment:
# print(build_candidate(REVIEWED_QUEUE_PATH))

## 13. D07 evaluation benchmark template — never train on D07

D07 must be independently curated, source-disjoint from training, and reviewed by domain experts. This notebook creates a blank template only; it does not fabricate benchmark questions or claim an evaluation benchmark already exists.

In [ ]:
d07 = pd.DataFrame(columns=[
 "benchmark_id","task_family","question","gold_answer_or_rubric",
 "source_or_basis","expected_verifier","difficulty",
 "contamination_check","expert_review_status","notes"
])
d07.to_csv(CANDIDATES / "D07_evaluation_template_DO_NOT_TRAIN.csv",index=False)
(REPORTS / "D07_rules.json").write_text(json.dumps({
 "training_use":False,"source_disjoint_from_training":True,
 "expert_review_required":True,"gold_or_rubric_required":True,
 "immutable_after_release":True
},indent=2))

## 14. Reconciliation report

Compare source counts, normalized counts and exact duplicate counts. The historical 596-record discrepancy is resolved only when the current source-to-normalized arithmetic and record-level dispositions fully account for every row.

In [ ]:
rows=[]
for name in ["D01","D02","D03","D04"]:
    source_n=len(raw.get(name,[])); norm_n=len(normalized.get(name,[]))
    dup_n=int(pool.loc[pool.source_dataset==name,"is_exact_duplicate"].sum()) if len(pool) else 0
    rows.append({"dataset":name,"source_rows":source_n,"normalized_rows":norm_n,
      "exact_duplicate_rows":dup_n,"source_minus_normalized":source_n-norm_n,
      "role":"retrieval_only" if name=="D03" else ("quarantined" if name=="D04" else "candidate_after_review")})
recon=pd.DataFrame(rows)
display(recon)
recon.to_csv(REPORTS/"count_reconciliation.csv",index=False)
print("SFT source total:",sum(len(raw.get(k,[])) for k in ["D01","D02","D04"]))
print("Normalized SFT total:",sum(len(normalized.get(k,[])) for k in ["D01","D02","D04"]))
print("Unique exact fingerprints:",int((~pool.is_exact_duplicate).sum()) if len(pool) else 0)

## 15. Export audit bundle

The ZIP is for inspection and feedback. It does not publish anything to Hugging Face or GitHub.

In [ ]:
metadata = {
 "run_id":RUN_ID,"created_utc":datetime.now(timezone.utc).isoformat(),
 "source_repositories":SOURCE_REPOS,"load_errors":load_errors,
 "allow_training":ALLOW_TRAINING,"allow_publish":ALLOW_PUBLISH,"auto_accept":AUTO_ACCEPT,
 "loaded_rows":{k:len(v) for k,v in raw.items()},
 "policy":["Original datasets unchanged","D03 retrieval-only","D04 quarantined pending trace replay",
           "D07 never used for training","No training or publishing performed"]
}
(REPORTS/"run_metadata.json").write_text(json.dumps(metadata,indent=2))
zip_path=OUT/f"trimorious_m1_2_audit_bundle_{RUN_ID}.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and f!=zip_path: z.write(f,f.relative_to(OUT))
print("Audit bundle:",zip_path)

## 16. M1.2 exit criteria

- [ ] All source rows accounted for; no unexplained count discrepancy.
- [ ] Schema mappings checked against raw examples.
- [ ] Exact duplicates resolved; near-duplicate candidates reviewed.
- [ ] D01 context/task mismatch cases reviewed and corrected or rejected.
- [ ] D02 task classes reviewed; applicable math independently checked.
- [ ] D03 provenance and licensing reviewed; retrieval-only policy preserved.
- [ ] D04 traces replayed or excluded.
- [ ] Human review decisions completed and retained.
- [ ] Train/validation split source-disjoint and inspected for semantic leakage.
- [ ] D07 independently curated, held out and never used for training.
- [ ] Dataset cards, provenance manifest and version identifier prepared.

**M1.2 deliverable:** a human-reviewed, versioned candidate bundle and audit report. Model training and publication are separate milestones requiring explicit approval.